# Expert Question Answerer for PDF



In [1]:

from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr

In [2]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

데이터 전처리

In [6]:
! uv pip install langchain-community pypdf

Using Python 3.12.12 environment at: C:\Users\jino7\Desktop\llm_engineering-main\.venv
Resolved 49 packages in 273ms
Prepared 1 package in 35ms
Installed 1 package in 44ms
 + pypdf==6.12.2


In [7]:
# 단일 PDF 파일을 불러옵니다
import os
from langchain_community.document_loaders import PyPDFLoader

# PDF 파일 경로 지정
PDF_PATH = "test.pdf"  # 실제 PDF 파일 경로로 변경하세요

# PyPDFLoader로 PDF 로드 (페이지별로 Document 생성)
loader = PyPDFLoader(PDF_PATH)
documents = loader.load()

# 각 페이지에 메타데이터 추가
for doc in documents:
    doc.metadata["source_file"] = os.path.basename(PDF_PATH)

print(f"총 {len(documents)}페이지를 불러왔습니다")


총 11페이지를 불러왔습니다


In [8]:
# RecursiveCharacterTextSplitter를 사용하여 문서를 청크(조각)로 분할합니다
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 텍스트 분할기 설정
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,    # 각 청크의 최대 글자 수
    chunk_overlap=200   # 인접한 청크 간 겹치는 글자 수 (문맥 유지용)
)

# 불러온 문서들을 청크로 분할
chunks = text_splitter.split_documents(documents)

# 결과 출력
print(f"총 {len(chunks)}개의 청크로 분할되었습니다")
print(f"첫 번째 청크:\n\n{chunks[0]}")

총 10개의 청크로 분할되었습니다
첫 번째 청크:

page_content='2022 
경기 규칙' metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2022-09-02T03:22:38+09:00', 'author': 'user', 'moddate': '2022-09-02T03:22:38+09:00', 'source': 'test.pdf', 'total_pages': 11, 'page': 0, 'page_label': '1', 'source_file': 'test.pdf'}


In [9]:
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
import os
DB_NAME = "pdf_vector_db_small"
#embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)
if os.path.exists(DB_NAME):
    Chroma(persist_directory=DB_NAME, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=DB_NAME)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

Vectorstore created with 10 documents


### These LangChain objects implement the method `invoke()`

벡터 스토어에 질문하는것

In [10]:
retriever = vectorstore.as_retriever()

retriever.invoke("경기 규칙 설명?")

[Document(id='f9919bb1-57c4-406e-8263-45ba0897ba30', metadata={'moddate': '2022-09-02T03:22:38+09:00', 'author': 'user', 'creationdate': '2022-09-02T03:22:38+09:00', 'page_label': '1', 'source_file': 'test.pdf', 'total_pages': 11, 'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'source': 'test.pdf', 'page': 0}, page_content='2022 \n경기 규칙'),
 Document(id='f9e19651-98fb-4445-b7c6-9fdc715d954f', metadata={'source': 'test.pdf', 'source_file': 'test.pdf', 'moddate': '2022-09-02T03:22:38+09:00', 'creator': 'Microsoft® Word 2019', 'total_pages': 11, 'page_label': '10', 'author': 'user', 'page': 9, 'producer': 'Microsoft® Word 2019', 'creationdate': '2022-09-02T03:22:38+09:00'}, page_content='LGFA PLAYING Rules 2022 8 \n \n \n경기 규칙 \n1. 선수는 서 있는 자세에서 발가락, 또는 한 손이나 두 \n손으로 그라운드에 있는 공을 집어 들 수 있습니다. \n \n2. 그라운드에 있는 선수는 자신으로부터 멀리 공을 차거나 \n보낼 수 있지만, 자신이 차지하기 위해 공을 가져올 수는 \n없습니다. \n \n3. 공을 잡으면 발로 차거나, 주먹으로 치거나, 손바닥으로 \n때리거나, 한 손이나 양손으로 한 번 튕기거나, 발끝에서 \n손으로 솔로할 수 있습니다. \n \n4

ChatGPT에게 질문 하는것

In [11]:
MODEL = "gpt-4.1-nano"

# temperature=0 means: always select the token with highest probability
#- temperature=1 usually means: a token with 10% probability should be picked 10% of the time
llm = ChatOpenAI(temperature=0, model_name=MODEL)

In [12]:
llm.invoke("경기 규칙 설명?")

AIMessage(content='경기 규칙은 스포츠 또는 게임의 종류에 따라 다르기 때문에 구체적인 종목을 알려주시면 더 정확한 설명을 드릴 수 있습니다. 예를 들어, 축구, 농구, 야구, 배드민턴, 탁구 등 각각의 경기 규칙이 다르기 때문입니다. \n\n일반적으로 경기 규칙은 다음과 같은 내용을 포함합니다:\n\n1. **경기 목적**: 승리 조건이나 목표\n2. **경기 시간**: 경기의 시작과 종료 시간, 연장 규정 등\n3. **선수 구성**: 팀당 선수 수, 교체 규정 등\n4. **경기 진행 방법**: 득점 방법, 공격과 수비 규칙, 반칙 규정\n5. **심판 역할**: 규칙 위반 시 판정권과 처벌\n\n혹시 특정 스포츠 또는 게임에 대한 규칙 설명이 필요하시면 알려 주세요!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 202, 'prompt_tokens': 13, 'total_tokens': 215, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_96f2a865f6', 'id': 'chatcmpl-DnJiTphTRtiKjUPwFLSTUjrK5b7Sg', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e96c9-4473-7173-a2ee

## Time to put this together!

In [ ]:
SYSTEM_PROMPT_TEMPLATE = """
당신은 Insurellm 회사를 대표하는 친절하고 전문적인 어시스턴트입니다.
사용자와 Insurellm에 대해 대화하고 있습니다.
관련이 있는 경우, 주어진 컨텍스트를 활용하여 질문에 답변하세요.
답을 모를 경우에는 모른다고 말씀해 주세요.
컨텍스트:
{context}
"""

In [16]:
SYSTEM_PROMPT_TEMPLATE = """'질의응답' 과정의 자료가 아래에 주어집니다. 질문에 대해 아래의 순서로 답변하세요. 한국어로 답변하세요.
관련이 있는 경우, 주어진 컨텍스트를 활용하여 질문에 답변하세요.
답을 모를 경우에는 모른다고 말씀해 주세요.
1. 주어진 문서의 내용이 질문에 얼마나 적합한지 평가하세요.
2. 질문에 대한 답변을 출력하세요.
3. metadata page도 답하라
---
컨텍스트:
{context}
---
Answer:"""

하나의 함수로 벡터 조회후 결과를  chatGPT 에게 넘기여서 
chatgpt가 LAG 가 반영된 결과를 받음

In [17]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [18]:
answer_question("경기 규칙 설명?", [])

'1. 주어진 문서의 내용이 질문에 적합한지 평가:  \n경기 규칙에 대한 구체적인 설명이 포함되어 있으며, 규칙의 주요 내용과 경기 진행에 관한 규정들이 상세히 제시되어 있어 질문에 적합합니다.\n\n2. 질문에 대한 답변:  \n경기 규칙은 여자 게일식 축구의 경기 진행과 선수의 행동에 관한 규정들을 포함하고 있습니다. 예를 들어, 선수는 서 있는 자세에서 공을 집거나 차는 행위, 공을 잡거나 튕기는 방법, 공을 이동하는 방법, 공을 던질 수 없는 규칙, 신체 접촉 금지, 어깨싸움 금지 등이 명시되어 있습니다. 또한, 출혈이나 두부 손상 의심 선수의 교체 규정도 포함되어 있습니다.\n\n3. metadata page:  \n페이지 번호는 명시되어 있지 않으며, 규칙은 LGFA 2022 경기 규칙에 관한 자료입니다.'

## What could possibly come next? 😂

Who is Averi Lancaster?
how much is her salary?

In [19]:
gr.ChatInterface(answer_question).launch()

c:\Users\jino7\Desktop\llm_engineering-main\.venv\Lib\site-packages\gradio\chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


## Admit it - you thought RAG would be more complicated than that!!